# pre-processing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Đang bắt đầu quá trình tiền xử lý...")

# --- 1. Tải và kết hợp dữ liệu ---
try:
    df_train = pd.read_csv('../data/txt/train.csv')
    df_test = pd.read_csv('../data/txt/test.csv')
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file train.csv hoặc test.csv.")
    exit()

# Lưu Id của test set và SalePrice của train set
test_ids = df_test['Id']
train_ids = df_train['Id']

# Tách biến mục tiêu (SalePrice) và log-transform
# np.log1p(x) = log(1+x), an toàn khi x=0
y_train_log = np.log1p(df_train['SalePrice'])
df_train = df_train.drop(['Id', 'SalePrice'], axis=1)
df_test = df_test.drop('Id', axis=1)

# Gộp train và test để xử lý đồng nhất
df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
print(f"Kích thước dữ liệu gộp (trước xử lý): {df_all.shape}")

# --- 2. Xử lý dữ liệu thiếu (Imputation) ---

# 2.1. CÁC BIẾN CATEGORICAL (Điền "None" cho các trường hợp 'NA' có ý nghĩa)
# Dựa trên data_description.txt, NA ở các cột này nghĩa là "Không có"
cols_fill_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 
    'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 
    'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType'
]
for col in cols_fill_none:
    df_all[col] = df_all[col].fillna('None')

# 2.2. CÁC BIẾN CATEGORICAL KHÁC (Điền giá trị phổ biến nhất - mode)
# Electrical chỉ thiếu 1 giá trị
df_all['Electrical'] = df_all['Electrical'].fillna(df_all['Electrical'].mode()[0])
# Các cột phân loại khác
cols_fill_mode = ['MSZoning', 'Utilities', 'Functional', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'SaleType']
for col in cols_fill_mode:
    df_all[col] = df_all[col].fillna(df_all[col].mode()[0])

# 2.3. CÁC BIẾN SỐ (Numerical)
# LotFrontage: Điền bằng 0 (hoặc median, 0 cũng là một lựa chọn hợp lý)
df_all['LotFrontage'] = df_all['LotFrontage'].fillna(0)
# MasVnrArea: Điền 0 (vì MasVnrType là "None")
df_all['MasVnrArea'] = df_all['MasVnrArea'].fillna(0)
# GarageYrBlt: Điền 0 khi không có Garage
df_all['GarageYrBlt'] = df_all['GarageYrBlt'].fillna(0)
# Các biến SF (diện tích) khác: điền 0
cols_fill_zero = ['GarageCars', 'GarageArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']
for col in cols_fill_zero:
    df_all[col] = df_all[col].fillna(0)

print(f"Số lượng giá trị thiếu còn lại: {df_all.isnull().sum().sum()}")

# --- 3. Tối ưu (Feature Engineering & Encoding) ---

# 3.1. Chuyển đổi các biến thứ tự (Ordinal Features)
# Đây là bước "tối ưu" quan trọng, giúp mô hình hiểu rõ thứ tự
# Tạo một bản đồ (mapping) từ 'Ex' (Excellent) đến 'Po' (Poor)
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]

for col in ordinal_cols:
    df_all[col] = df_all[col].map(quality_map).fillna(0) # fillna(0) phòng trường hợp có giá trị lạ

# Một số mapping đặc biệt khác
df_all['BsmtExposure'] = df_all['BsmtExposure'].map({'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}).fillna(0)
df_all['LandSlope'] = df_all['LandSlope'].map({'Gtl': 3, 'Mod': 2, 'Sev': 1}).fillna(0)
df_all['LotShape'] = df_all['LotShape'].map({'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1}).fillna(0)
df_all['PavedDrive'] = df_all['PavedDrive'].map({'Y': 3, 'P': 2, 'N': 1}).fillna(0)
df_all['Utilities'] = df_all['Utilities'].map({'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1}).fillna(0)

# 3.2. Chuyển đổi các biến danh nghĩa (Nominal Features) - One-Hot Encoding
# Biến MSSubClass là số nhưng thực ra là categorical
df_all['MSSubClass'] = df_all['MSSubClass'].astype(str)

# Dùng get_dummies cho tất cả các cột 'object' còn lại
df_all_processed = pd.get_dummies(df_all, drop_first=True)

print(f"Kích thước dữ liệu sau khi One-Hot Encoding: {df_all_processed.shape}")

# --- 4. Tách lại và Scaling ---

# Tách lại train và test
X_train = df_all_processed.iloc[:len(y_train_log)]
X_test = df_all_processed.iloc[len(y_train_log):]

# 4.1. Scaling (Chuẩn hóa)
# Chỉ lấy các cột số (loại trừ các cột one-hot đã là 0/1)
numerical_cols = df_train.select_dtypes(include=np.number).columns
# Lọc ra các cột số có trong X_train (vì một số có thể đã bị map)
cols_to_scale = [col for col in numerical_cols if col in X_train.columns]

scaler = StandardScaler()

# CHỈ fit trên X_train (để tránh rò rỉ dữ liệu từ test set)
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

# DÙNG scaler đã fit để transform X_test
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("Đã hoàn thành Scaling!")

# --- 5. Lưu kết quả ---
X_train.to_csv('../data/txt-preprocessing/train_processed.csv', index=False)
X_test.to_csv('../data/txt-preprocessing/test_processed.csv', index=False)
y_train_log.to_csv('../data/txt-preprocessing/y_train_log.csv', index=False, header=['SalePrice_Log'])

print("\n--- HOÀN THÀNH TIỀN XỬ LÝ ---")
print("Đã lưu 3 files:")
print("1. train_processed.csv (Dữ liệu train đã xử lý)")
print("2. test_processed.csv (Dữ liệu test đã xử lý)")
print("3. y_train_log.csv (Biến mục tiêu đã log-transform)")

Đang bắt đầu quá trình tiền xử lý...
Kích thước dữ liệu gộp (trước xử lý): (2919, 79)
Số lượng giá trị thiếu còn lại: 0
Kích thước dữ liệu sau khi One-Hot Encoding: (2919, 236)
Đã hoàn thành Scaling!


C:\Users\pikal\AppData\Local\Temp\ipykernel_15076\383099030.py:109: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
C:\Users\pikal\AppData\Local\Temp\ipykernel_15076\383099030.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])



--- HOÀN THÀNH TIỀN XỬ LÝ ---
Đã lưu 3 files:
1. train_processed.csv (Dữ liệu train đã xử lý)
2. test_processed.csv (Dữ liệu test đã xử lý)
3. y_train_log.csv (Biến mục tiêu đã log-transform)
